# Question B: baseline AC model

IS4001 Smart Building Analytics project.

**What this script does**

For each air-conditioned zone on the chosen floor (Floor 4 by default), it trains a model that
predicts **normal hourly AC power** from outdoor weather and time, then checks whether the model
is accurate enough to use as a baseline.

1. Loads the clean hourly table made by `cubems_prepare` (weather and holidays already included).
2. Keeps occupied workday hours only (`is_occupied`): holidays, weekends and nights are excluded.
3. Splits time-wise: **train Jul 2018 - Jun 2019** (a full seasonal cycle), **test Jul - Dec 2019**.
4. Fits two models per zone:
   - **Linear regression** (primary, statsmodels):
     `AC kW ~ outdoor temp + outdoor RH + hour + day of week + day after holiday`
   - **Gradient boosting** (challenger: LightGBM, or scikit-learn's HistGradientBoosting if
     LightGBM isn't installed) on the same inputs.
5. Scores both on the test period with the **ASHRAE Guideline 14** hourly criteria:
   CV(RMSE) <= 30 % and NMBE within +/-10 %.
6. Picks the model per zone: linear regression unless the challenger is clearly better.
7. Checks for a step change between 2018 and 2019 (Floor 4's July AC use rose ~19 %).
8. Saves predictions and residuals (actual - predicted) for every occupied hour, ready for the
   EWMA anomaly flags and the comfort cross-check, plus coefficients, metrics and charts.

**Important:** the baseline describes how the building *normally* ran, overcooling included.
It is not a target. It answers "how much AC would this hour normally use, given the weather?",
so savings and anomalies are measured against business as usual.

**Requirements:** `pip install pandas numpy statsmodels scikit-learn matplotlib joblib`
(optional: `lightgbm`).

## 1. Settings

In [2]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import joblib

DATA_DIR = Path("../../data/qns2/processed")          # OUT_DIR of cubems_prepare
OUT_DIR = Path("../../data/qns2/baseline")
FLOOR = 4                                  # primary floor; run again with 5 for the replication check

TRAIN_START, TRAIN_END = "2018-07-01", "2019-06-30"
TEST_START, TEST_END = "2019-07-01", "2019-12-31"

# Inputs. Indoor temperature is deliberately NOT an input: it is mostly caused by the AC
# itself, so using it would make the model circular (and it is missing Sep 2018 - Mar 2019).
WEATHER_FEATURES = ["out_temp_c", "out_rh_pct"]
LINEAR_FORMULA = ("ac_kw ~ out_temp_c + out_rh_pct + C(hour) + C(day_of_week)"
                  " + day_after_holiday")

# ASHRAE Guideline 14, hourly calibration criteria
CVRMSE_LIMIT, NMBE_LIMIT = 30.0, 10.0
# The challenger replaces linear regression only if it passes ASHRAE 14 and beats the linear
# model's test CV(RMSE) by at least this many percentage points.
CHALLENGER_MARGIN = 2.0

# A zone needs at least this many occupied training hours (about 4 months of workdays).
MIN_TRAIN_HOURS = 800
# Zones whose AC averages below this during occupied hours are skipped (effectively unused).
MIN_MEAN_KW = 0.5

# Step-change check: flag if the mean residual differs between Jul-Dec 2018 and Jan-Jun 2019
# by more than this share of mean AC power.
STEP_CHANGE_THRESHOLD = 0.10
# Set True to add a 2019 indicator to the linear model if the check above flags a step change.
ADD_PERIOD_INDICATOR = False

RANDOM_SEED = 42

## 2. Load data

In [3]:
def load_floor(floor: int) -> pd.DataFrame:
    path = DATA_DIR / "clean_hourly" / f"floor{floor}_hourly.csv"
    parquet = path.with_suffix(".parquet")
    if parquet.exists():
        df = pd.read_parquet(parquet)
        if "timestamp" in df.columns:
            df = df.set_index("timestamp")
    else:
        df = pd.read_csv(path, parse_dates=["timestamp"], index_col="timestamp")
    for col in ["is_occupied", "is_holiday", "is_workday", "day_after_holiday", "is_weekend"]:
        if col in df:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return df.sort_index()


def ac_zones(df: pd.DataFrame) -> list[str]:
    return sorted(c.replace("_ac_total_kw", "") for c in df.columns if c.endswith("_ac_total_kw"))


def zone_frame(df: pd.DataFrame, zone: str) -> pd.DataFrame:
    """Occupied hours with a valid target and weather, in the columns the models use."""
    cols = ["hour", "day_of_week", "day_after_holiday", "month"] + WEATHER_FEATURES
    z = df.loc[df["is_occupied"], cols].copy()
    z["ac_kw"] = df.loc[df["is_occupied"], f"{zone}_ac_total_kw"]
    z["day_after_holiday"] = z["day_after_holiday"].astype(int)
    z["period_2019"] = (z.index.year >= 2019).astype(int)
    z["split"] = np.select(
        [(z.index >= TRAIN_START) & (z.index <= f"{TRAIN_END} 23:59"),
         (z.index >= TEST_START) & (z.index <= f"{TEST_END} 23:59")],
        ["train", "test"], "other")
    return z.dropna(subset=["ac_kw"] + WEATHER_FEATURES)

## 3. Accuracy metrics (ASHRAE Guideline 14)
CV(RMSE) measures typical error size relative to average use. NMBE measures whether the model
is biased high or low overall. `p` is the number of model parameters (training fit);
for the test period p = 1 is used, as is usual for out-of-sample checks.

In [4]:
def ashrae_metrics(actual: pd.Series, predicted: pd.Series, p: int = 1) -> dict:
    y, yhat = np.asarray(actual, float), np.asarray(predicted, float)
    n, mean = len(y), y.mean()
    dof = max(n - p, 1)
    rmse_dof = np.sqrt(np.sum((y - yhat) ** 2) / dof)
    ss_res, ss_tot = np.sum((y - yhat) ** 2), np.sum((y - mean) ** 2)
    return {
        "n_hours": n,
        "mean_kw": round(mean, 3),
        "cv_rmse_pct": round(rmse_dof / mean * 100, 2) if mean else np.nan,
        "nmbe_pct": round(np.sum(y - yhat) / (dof * mean) * 100, 2) if mean else np.nan,
        "r2": round(1 - ss_res / ss_tot, 3) if ss_tot else np.nan,
        "mae_kw": round(np.mean(np.abs(y - yhat)), 3),
    }


def passes_ashrae(m: dict) -> bool:
    return m["cv_rmse_pct"] <= CVRMSE_LIMIT and abs(m["nmbe_pct"]) <= NMBE_LIMIT

## 4. Models

In [5]:
def fit_linear(train: pd.DataFrame, formula: str):
    return smf.ols(formula, data=train).fit()


def make_challenger():
    try:
        from lightgbm import LGBMRegressor
        return "LightGBM", LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15,
                                         min_child_samples=30, subsample=0.8, subsample_freq=1,
                                         colsample_bytree=0.9, random_state=RANDOM_SEED, verbose=-1)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        return "HistGradientBoosting", HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=30,
            random_state=RANDOM_SEED)


CHALLENGER_FEATURES = ["out_temp_c", "out_rh_pct", "hour", "day_of_week", "day_after_holiday"]


def fit_challenger(train: pd.DataFrame):
    name, model = make_challenger()
    model.fit(train[CHALLENGER_FEATURES], train["ac_kw"])
    return name, model


def step_change_check(z: pd.DataFrame, linear_model) -> dict:
    """Mean training residual in Jul-Dec 2018 vs Jan-Jun 2019, relative to mean AC power."""
    train = z[z["split"] == "train"]
    resid = train["ac_kw"] - linear_model.predict(train)
    h2_2018 = resid[train.index.year == 2018].mean()
    h1_2019 = resid[train.index.year == 2019].mean()
    gap = (h1_2019 - h2_2018) / train["ac_kw"].mean()
    return {"resid_mean_2018h2_kw": round(h2_2018, 3), "resid_mean_2019h1_kw": round(h1_2019, 3),
            "step_change_share": round(gap, 3), "step_change_flag": bool(abs(gap) > STEP_CHANGE_THRESHOLD)}

## 5. Charts

In [6]:
def plot_zone(zone: str, z: pd.DataFrame, chosen: str, out_dir: Path) -> None:
    test = z[z["split"] == "test"]
    fig, axes = plt.subplots(3, 1, figsize=(11, 10))

    # (a) two sample test weeks: actual vs predicted
    week = test.loc[test.index.min(): test.index.min() + pd.Timedelta(days=13)]
    ax = axes[0]
    ax.plot(week.index, week["ac_kw"], ".", ms=4, label="Actual", color="#444444")
    ax.plot(week.index, week["pred_linear"], ".", ms=4, label="Linear", color="#1f77b4")
    ax.plot(week.index, week["pred_challenger"], ".", ms=4, label="Challenger", color="#ff7f0e", alpha=0.7)
    ax.set_title(f"Floor {FLOOR} {zone}: first two test weeks (occupied hours)")
    ax.set_ylabel("AC power (kW)")
    ax.legend(loc="upper right", frameon=False)

    # (b) actual vs predicted, chosen model, test period
    ax = axes[1]
    pred = test[f"pred_{chosen}"]
    ax.scatter(pred, test["ac_kw"], s=6, alpha=0.4, color="#1f77b4")
    lim = [0, max(pred.max(), test["ac_kw"].max()) * 1.05]
    ax.plot(lim, lim, color="#888888", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Predicted AC power (kW)"); ax.set_ylabel("Actual AC power (kW)")
    ax.set_title(f"Test period, {chosen} model (points on the line = perfect prediction)")

    # (c) mean residual by month, chosen model, whole period
    ax = axes[2]
    monthly = z["resid"].groupby(z.index.to_period("M")).mean()
    colors = ["#1f77b4" if p.to_timestamp() <= pd.Timestamp(TRAIN_END) else "#ff7f0e" for p in monthly.index]
    ax.bar(monthly.index.astype(str), monthly.values, color=colors)
    ax.axhline(0, color="#888888", lw=1)
    ax.set_ylabel("Mean residual (kW)")
    ax.set_title("Actual minus predicted by month (blue = training, orange = test)")
    ax.tick_params(axis="x", rotation=60)

    fig.tight_layout()
    fig.savefig(out_dir / f"floor{FLOOR}_{zone}_baseline.png", dpi=130)
    plt.close(fig)

## 6. Run

In [7]:
def main() -> dict:
    out = OUT_DIR / f"floor{FLOOR}"
    (out / "models").mkdir(parents=True, exist_ok=True)
    (out / "charts").mkdir(parents=True, exist_ok=True)

    df = load_floor(FLOOR)
    zones = ac_zones(df)
    print(f"Floor {FLOOR}: AC zones found {zones}")

    metrics_rows, coef_rows, pred_frames, checks = [], [], [], []
    for zone in zones:
        z = zone_frame(df, zone)
        train, test = z[z["split"] == "train"], z[z["split"] == "test"]
        if len(train) < MIN_TRAIN_HOURS or z["ac_kw"].mean() < MIN_MEAN_KW:
            print(f"  {zone}: skipped ({len(train)} training hours, mean {z['ac_kw'].mean():.2f} kW)")
            continue

        # Linear regression, with an optional 2019 indicator if a step change is found
        formula = LINEAR_FORMULA
        linear = fit_linear(train, formula)
        check = {"zone": zone, **step_change_check(z, linear)}
        if check["step_change_flag"] and ADD_PERIOD_INDICATOR:
            formula = LINEAR_FORMULA + " + period_2019"
            linear = fit_linear(train, formula)
            check["period_indicator_added"] = True
        checks.append(check)

        challenger_name, challenger = fit_challenger(train)

        z["pred_linear"] = linear.predict(z).clip(lower=0)
        z["pred_challenger"] = np.clip(challenger.predict(z[CHALLENGER_FEATURES]), 0, None)

        results = {}
        for model_name, col, p in [("linear", "pred_linear", int(linear.df_model) + 1),
                                   ("challenger", "pred_challenger", 1)]:
            for split, part in [("train", train), ("test", test)]:
                m = ashrae_metrics(part["ac_kw"], z.loc[part.index, col], p if split == "train" else 1)
                m.update({"zone": zone, "model": challenger_name if model_name == "challenger" else "Linear regression",
                          "role": model_name, "split": split, "passes_ashrae14": passes_ashrae(m)})
                metrics_rows.append(m)
                results[(model_name, split)] = m

        lin_t, ch_t = results[("linear", "test")], results[("challenger", "test")]
        chosen = "challenger" if (passes_ashrae(ch_t) and
                                  ch_t["cv_rmse_pct"] <= lin_t["cv_rmse_pct"] - CHALLENGER_MARGIN) else "linear"
        z["chosen_model"] = chosen
        z["pred_chosen"] = z[f"pred_{chosen}"]
        z["resid"] = z["ac_kw"] - z["pred_chosen"]
        z["zone"] = zone
        pred_frames.append(z)

        coefs = pd.DataFrame({"coef": linear.params, "std_err": linear.bse, "p_value": linear.pvalues})
        coefs.insert(0, "zone", zone)
        coef_rows.append(coefs.reset_index(names="term"))

        joblib.dump({"linear": linear, "challenger": challenger, "formula": formula,
                     "challenger_features": CHALLENGER_FEATURES, "chosen": chosen},
                    out / "models" / f"floor{FLOOR}_{zone}_baseline.joblib")
        plot_zone(zone, z, chosen, out / "charts")

        status = "PASS" if passes_ashrae(results[(chosen, "test")]) else "FAIL"
        print(f"  {zone}: linear test CV(RMSE) {lin_t['cv_rmse_pct']:.1f}% NMBE {lin_t['nmbe_pct']:+.1f}% | "
              f"{challenger_name} {ch_t['cv_rmse_pct']:.1f}% / {ch_t['nmbe_pct']:+.1f}% -> using {chosen} "
              f"(ASHRAE 14 {status})")
        if check["step_change_flag"]:
            print(f"    step change between 2018 and 2019: {check['step_change_share']:+.0%} of mean AC power"
                  + ("" if ADD_PERIOD_INDICATOR else " (set ADD_PERIOD_INDICATOR = True to model it)"))

    if not pred_frames:
        raise RuntimeError("No zone had enough data to train a baseline.")

    metrics = pd.DataFrame(metrics_rows)[["zone", "model", "role", "split", "n_hours", "mean_kw",
                                          "cv_rmse_pct", "nmbe_pct", "r2", "mae_kw", "passes_ashrae14"]]
    coefs = pd.concat(coef_rows, ignore_index=True)
    preds = pd.concat(pred_frames)[["zone", "split", "ac_kw", "pred_linear", "pred_challenger",
                                    "chosen_model", "pred_chosen", "resid"] + WEATHER_FEATURES
                                   + ["hour", "day_of_week", "month"]]
    preds.index.name = "timestamp"

    metrics.to_csv(out / "metrics.csv", index=False)
    coefs.to_csv(out / "linear_coefficients.csv", index=False)
    preds.to_csv(out / "predictions_occupied_hours.csv", float_format="%.4f")
    pd.DataFrame(checks).to_csv(out / "step_change_check.csv", index=False)
    (out / "settings.json").write_text(json.dumps({
        "floor": FLOOR, "train": [TRAIN_START, TRAIN_END], "test": [TEST_START, TEST_END],
        "linear_formula": LINEAR_FORMULA, "challenger_features": CHALLENGER_FEATURES,
        "ashrae14_limits": {"cv_rmse_pct": CVRMSE_LIMIT, "nmbe_pct": NMBE_LIMIT},
        "challenger_margin_pct_points": CHALLENGER_MARGIN}, indent=2))

    # Plain-language summary of the temperature effect
    temp = coefs[coefs["term"] == "out_temp_c"][["zone", "coef", "p_value"]]
    print("\nLinear model: extra AC power per +1 °C outdoors (occupied hours)")
    for _, r in temp.iterrows():
        print(f"  {r['zone']}: {r['coef']:+.2f} kW per °C (p = {r['p_value']:.3g})")
    print(f"\nSaved to {out.resolve()}")
    return {"metrics": metrics, "coefficients": coefs, "predictions": preds}


with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    results = main()

Floor 4: AC zones found ['z1', 'z2', 'z4', 'z5']


c:\Users\ganad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ganad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\ganad\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ganad\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\ganad\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

  z1: linear test CV(RMSE) 24.4% NMBE +1.8% | HistGradientBoosting 25.4% / +2.9% -> using linear (ASHRAE 14 PASS)
  z2: linear test CV(RMSE) 21.0% NMBE +8.4% | HistGradientBoosting 20.1% / +7.5% -> using linear (ASHRAE 14 PASS)
  z4: linear test CV(RMSE) 19.0% NMBE -1.5% | HistGradientBoosting 19.9% / -0.7% -> using linear (ASHRAE 14 PASS)
  z5: linear test CV(RMSE) 24.2% NMBE +9.8% | HistGradientBoosting 24.9% / +10.6% -> using linear (ASHRAE 14 PASS)

Linear model: extra AC power per +1 °C outdoors (occupied hours)
  z1: +1.11 kW per °C (p = 1.3e-104)
  z2: +0.26 kW per °C (p = 2.18e-14)
  z4: +0.82 kW per °C (p = 2.73e-78)
  z5: +0.38 kW per °C (p = 1.87e-67)

Saved to C:\Users\ganad\Documents\GitHub\IS4001-G1T4\data\qns2\baseline\floor4


## 7. Next steps

`predictions_occupied_hours.csv` has `resid` (actual minus predicted) for every occupied hour.
- **Anomaly flags:** run an EWMA control chart on `resid` per zone; sustained positive runs are
  hours where the zone used more AC than normal for the weather.
- **Comfort cross-check:** join those hours to PMV from the comfort analysis to label each flag
  too cool / comfortable / too warm.
- **Measuring savings later:** after a setpoint change, savings = predicted minus actual.